# CRISP-DM: Telecom Customer Churn

This Colab follows Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, and Deployment.

In [1]:
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, ConfusionMatrixDisplay
RANDOM_STATE=42
DATA_PATH='data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
assert os.path.exists(DATA_PATH), 'Download the Kaggle CSV; see ../data/README.md'
df=pd.read_csv(DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Business understanding

The objective is to rank customers by churn risk so retention teams can prioritize outreach. We report recall, precision, ROC-AUC, and PR-AUC because the intervention budget makes threshold choice important.

In [2]:
df.info()
display(df['Churn'].value_counts(normalize=True))
display(df.isna().sum().sort_values(ascending=False).head())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

customerID       0
gender           0
SeniorCitizen    0
Partner          0
Dependents       0
dtype: int64

## 2–3. Data understanding and preparation

In [3]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'], errors='coerce')
df=df.drop(columns=['customerID'])
X=df.drop(columns=['Churn']); y=(df['Churn']=='Yes').astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
num=X.select_dtypes(include=np.number).columns.tolist()
cat=X.select_dtypes(exclude=np.number).columns.tolist()
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])

## 4–5. Modeling and evaluation

The logistic model is an interpretable baseline. The random forest tests whether nonlinear interactions improve ranking.

In [4]:
models={'logistic':LogisticRegression(max_iter=1000,class_weight='balanced',random_state=RANDOM_STATE),'random_forest':RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1)}
results=[]
for name,model in models.items():
    pipe=Pipeline([('preprocess',pre),('model',model)])
    pipe.fit(X_train,y_train)
    p=pipe.predict_proba(X_test)[:,1]
    pred=(p>=.5).astype(int)
    results.append({'model':name,'roc_auc':roc_auc_score(y_test,p),'pr_auc':average_precision_score(y_test,p)})
    print(name); print(classification_report(y_test,pred,target_names=['stay','churn']))
display(pd.DataFrame(results))

logistic
              precision    recall  f1-score   support

        stay       0.90      0.72      0.80      1035
       churn       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



random_forest
              precision    recall  f1-score   support

        stay       0.86      0.81      0.84      1035
       churn       0.55      0.62      0.58       374

    accuracy                           0.76      1409
   macro avg       0.70      0.72      0.71      1409
weighted avg       0.77      0.76      0.77      1409



,model,roc_auc,pr_auc
0,logistic,0.841298,0.632578
1,random_forest,0.821759,0.603780


## 6. Deployment

A weekly batch job would score active customers and send the top-risk segment to a retention queue. Production monitoring should track feature drift, score drift, recall after labels mature, calibration, intervention rate, and fairness across relevant groups. The model is a decision-support tool, not an automatic adverse decision.